# Steam Games Cleaning Pipeline

## 1. Setup đường dẫn

In [ ]:
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_PATH = RAW_DIR / "SteamGames.csv"
PROCESSED_PATH = PROCESSED_DIR / "SteamGames_cleaned.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("PROCESSED_PATH:", PROCESSED_PATH)


## 2. Import thư viện

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd
import seaborn as sns
import re

import json
import time


## 3. Đọc dữ liệu

In [ ]:
# Đường dẫn Kaggle hiện tại của bạn. Nếu chạy theo project structure, dùng RAW_PATH.
kaggle_path = Path('/kaggle/input/datasets/newnguyn/steam-game-clean/SteamGames.csv')
df = pd.read_csv(kaggle_path if kaggle_path.exists() else RAW_PATH)


## 4. Chuẩn hóa missing thô, xử lý duplicate `Appid` và đánh lại `Rank`

Bước này giải quyết lỗi quan trọng nhất trong dữ liệu thô: cùng một `Appid` có thể xuất hiện nhiều lần do crawl ngắt quãng hoặc rank bị trùng.

Nguyên tắc xử lý:

- `Appid` là khóa định danh của game trên Steam nên dataset cuối **phải unique theo Appid**.
- Với các Appid bị trùng, giữ bản ghi **giàu thông tin hơn** trước, nếu hòa thì giữ bản ghi có `Rank` nhỏ hơn, sau đó giữ thứ tự crawl ban đầu.
- Sau khi deduplicate, đánh lại `Rank` liên tục từ 1 đến N để báo cáo không còn mâu thuẫn về rank.
- Xuất file audit các Appid trùng để có bằng chứng nếu thầy hỏi.


In [ ]:
# ============================================================
# RAW AUDIT + DEDUPLICATE BY APPID + RANK REALIGNMENT
# ============================================================

# Lưu snapshot raw để đối chiếu báo cáo.
df_raw_snapshot = df.copy(deep=True)

QUALITY_REPORT_DIR = PROCESSED_DIR / "quality_reports"
QUALITY_REPORT_DIR.mkdir(parents=True, exist_ok=True)

cleaning_step_stats = []

def count_empty_string_cells(dataframe):
    if dataframe.empty:
        return 0
    obj_cols = dataframe.select_dtypes(include="object").columns
    if len(obj_cols) == 0:
        return 0
    return int(
        dataframe[obj_cols]
        .apply(lambda s: s.astype(str).str.strip().eq("").sum())
        .sum()
    )

def log_cleaning_step(step, dataframe, note=""):
    duplicate_appid = (
        int(dataframe["Appid"].duplicated().sum())
        if "Appid" in dataframe.columns
        else None
    )
    cleaning_step_stats.append({
        "step": step,
        "rows": int(len(dataframe)),
        "columns": int(dataframe.shape[1]),
        "missing_cells": int(dataframe.isna().sum().sum()),
        "empty_string_cells": count_empty_string_cells(dataframe),
        "duplicate_appid": duplicate_appid,
        "note": note,
    })

def standardize_blank_to_nan(dataframe, columns=None):
    # Chuẩn hóa chuỗi rỗng/toàn khoảng trắng thành NaN.
    # Làm sớm để fill/drop không bị sai do "" không được pandas xem là NaN.
    target_cols = columns if columns is not None else dataframe.select_dtypes(include="object").columns
    for col in target_cols:
        if col in dataframe.columns:
            dataframe[col] = dataframe[col].replace(r"^\s*$", np.nan, regex=True)
    return dataframe

log_cleaning_step("00_raw_input", df, "Original crawled dataset before cleaning.")

# Chuẩn hóa tên cột phòng trường hợp CSV có khoảng trắng thừa.
df.columns = df.columns.astype(str).str.strip()

required_columns = ["Appid", "Name", "ReleaseDate", "Developers", "Publishers", "Tags"]
missing_required_columns = [col for col in required_columns if col not in df.columns]
if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

# Ép Appid về numeric để khóa định danh nhất quán.
df["Appid"] = pd.to_numeric(df["Appid"], errors="coerce")
df = df.dropna(subset=["Appid"]).copy()
df["Appid"] = df["Appid"].astype("int64")

# Chuẩn hóa blank string thành NaN ngay từ đầu.
df = standardize_blank_to_nan(df)

# Audit duplicate Appid trước khi xử lý.
duplicate_appid_before = int(df["Appid"].duplicated().sum())
duplicate_rows_before = int(df.duplicated().sum())

df["_raw_order"] = np.arange(len(df))
df["_completeness_score"] = df.notna().sum(axis=1)

if "Rank" in df.columns:
    df["_rank_numeric"] = pd.to_numeric(df["Rank"], errors="coerce").fillna(np.inf)
else:
    df["_rank_numeric"] = np.inf

duplicate_audit = (
    df[df["Appid"].duplicated(keep=False)]
    .sort_values(["Appid", "_completeness_score", "_rank_numeric", "_raw_order"],
                 ascending=[True, False, True, True])
    .copy()
)

duplicate_audit_path = QUALITY_REPORT_DIR / "duplicate_appid_audit_before_dedup.csv"
duplicate_audit.drop(
    columns=[c for c in ["_raw_order", "_completeness_score", "_rank_numeric"] if c in duplicate_audit.columns],
    errors="ignore"
).to_csv(duplicate_audit_path, index=False, encoding="utf-8-sig")

# Deduplicate:
# - Ưu tiên dòng có nhiều field không thiếu nhất.
# - Nếu hòa thì ưu tiên Rank nhỏ hơn.
# - Nếu vẫn hòa thì ưu tiên dòng xuất hiện sớm hơn trong quá trình crawl.
df = (
    df.sort_values(
        ["Appid", "_completeness_score", "_rank_numeric", "_raw_order"],
        ascending=[True, False, True, True]
    )
    .drop_duplicates(subset=["Appid"], keep="first")
    .sort_values("_raw_order")
    .drop(columns=["_raw_order", "_completeness_score", "_rank_numeric"])
    .reset_index(drop=True)
)

# Rank cũ chỉ phản ánh trạng thái crawl bị gối đầu, nên đánh lại liên tục.
df["Rank"] = np.arange(1, len(df) + 1)

# Assert: Appid và Rank phải unique sau bước này.
assert df["Appid"].is_unique, "Appid vẫn còn trùng sau deduplication."
assert df["Rank"].is_unique and df["Rank"].min() == 1 and df["Rank"].max() == len(df), "Rank chưa liên tục sau realignment."

dedup_summary = pd.DataFrame([
    {
        "metric": "raw_rows_before_dedup",
        "value": len(df_raw_snapshot),
        "note": "Rows in original crawled file."
    },
    {
        "metric": "duplicate_appid_before_dedup",
        "value": duplicate_appid_before,
        "note": "Number of duplicated Appid occurrences before deduplication."
    },
    {
        "metric": "duplicate_rows_before_dedup",
        "value": duplicate_rows_before,
        "note": "Exact duplicate rows before cleaning."
    },
    {
        "metric": "rows_after_appid_dedup",
        "value": len(df),
        "note": "Rows after keeping one best record per Appid."
    },
    {
        "metric": "duplicate_appid_after_dedup",
        "value": int(df["Appid"].duplicated().sum()),
        "note": "Must be 0 for final dataset."
    },
])
dedup_summary_path = QUALITY_REPORT_DIR / "dedup_rank_summary.csv"
dedup_summary.to_csv(dedup_summary_path, index=False, encoding="utf-8-sig")

log_cleaning_step(
    "01_appid_dedup_rank_realignment",
    df,
    f"Deduplicated by Appid. Duplicate Appid before={duplicate_appid_before}, after=0. Rank realigned 1..N."
)

display(dedup_summary)
print("Saved duplicate Appid audit:", duplicate_audit_path)
print("Saved dedup summary:", dedup_summary_path)
print("Current shape after dedup:", df.shape)


## 5. Check NULL ban đầu

In [ ]:
df.isnull().sum()

## 6. Drop dòng thiếu `ReleaseDate` hoặc `Name`

In [ ]:
# ReleaseDate bắt buộc parse được datetime.
rows_before = len(df)

df["ReleaseDate"] = pd.to_datetime(df["ReleaseDate"], errors="coerce")
df = df.dropna(subset=["ReleaseDate"]).copy()

# Name NULL/rỗng thì drop, không fill.
df["Name"] = df["Name"].replace(r"^\s*$", np.nan, regex=True)
df = df.dropna(subset=["Name"]).copy()
df["Name"] = df["Name"].astype(str).str.strip()
df = df[df["Name"].ne("")].copy()

df = df.reset_index(drop=True)

log_cleaning_step(
    "02_drop_missing_release_date_or_name",
    df,
    f"Dropped {rows_before - len(df)} rows with invalid ReleaseDate or missing/empty Name."
)

display(df[["Name", "ReleaseDate"]].isnull().sum())
print("Rows before:", rows_before, "| after:", len(df), "| dropped:", rows_before - len(df))


## 7. Fill `Developers` và `Publishers` qua lại

In [ ]:
# Chuẩn hóa chuỗi rỗng/toàn khoảng trắng thành NaN trước khi fill qua lại.
# Nếu fill trước khi xử lý "" thì pandas không coi "" là thiếu, dễ drop nhầm dòng còn Publisher/Developer hợp lệ.
for col in ["Developers", "Publishers"]:
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

# Developers và Publishers thường có thể thay thế nhau trong dữ liệu Steam.
df["Publishers"] = df["Publishers"].fillna(df["Developers"])
df["Developers"] = df["Developers"].fillna(df["Publishers"])



### 7.1. Check NULL sau khi fill dev/pub

In [ ]:
df.isnull().sum()   

### 7.2. Drop dòng thiếu cả dev/pub

In [ ]:
# Những game sau khi fill qua lại mà vẫn thiếu Developers hoặc Publishers thì không rõ nguồn gốc,
# nên loại khỏi dataset khuyến nghị.
rows_before = len(df)

for col in ["Developers", "Publishers"]:
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

# Fill lại một lần nữa phòng trường hợp vừa convert "" -> NaN ở cell này.
df["Publishers"] = df["Publishers"].fillna(df["Developers"])
df["Developers"] = df["Developers"].fillna(df["Publishers"])

df = df.dropna(subset=["Developers", "Publishers"]).copy()
df = df.reset_index(drop=True)

log_cleaning_step(
    "03_fill_cross_dev_pub_and_drop_unresolved",
    df,
    f"Dropped {rows_before - len(df)} rows still missing both Developers/Publishers after cross-fill."
)

display(df.isnull().sum())
print("Rows before:", rows_before, "| after:", len(df), "| dropped:", rows_before - len(df))


## 8. Check Description thiếu

In [ ]:
display(df[df['Description'].isnull()])

### 8.1. Fill Description

In [ ]:
# Nhận thấy có khá nhiều game không có phần mô tả, tuy nhiên ta vẫn có thể sử dụng thông tin khác để khuyến nghị game nên ta sẽ fillna "No description" cho những game này
df['Description'] = df['Description'].fillna("No description")

In [ ]:
df.isnull().sum()

## 9. Fill requirement text thiếu

In [ ]:
# Với những thuộc tính yêu cầu về OS, Memory, CPU:
# - NaN hoặc chuỗi rỗng được xem là không khai báo yêu cầu rõ ràng.
# - Fill bằng "No requirement" để không phát sinh NULL khi phân tích/filter.
for col in ["OsRequirement", "MemoryRequirement", "CpuRequirement"]:
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)
    df[col] = df[col].fillna("No requirement")

log_cleaning_step(
    "04_fill_requirement_text",
    df,
    "Filled missing/blank OS, memory and CPU requirement fields with 'No requirement'."
)

display(df[["OsRequirement", "MemoryRequirement", "CpuRequirement"]].isna().sum())


In [ ]:
df.isnull().sum()   

## 10. Fill `Tags` bằng `Genres` nếu thiếu

In [ ]:
# Chuẩn hóa chuỗi rỗng thành NaN trước khi dùng Genres làm fallback cho Tags.
# Nếu không làm bước này, Tags = "" sẽ không được fill bằng Genres và khi save CSV sẽ thành ô trống.
for col in ["Tags", "Genres"]:
    if col in df.columns:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

# Tags thiếu thì dùng Genres.
df["Tags"] = df["Tags"].fillna(df["Genres"])

# Nếu vẫn không có tag/genre thì dùng nhãn an toàn, không để chuỗi rỗng.
df["Tags"] = df["Tags"].fillna("no tags")



In [ ]:
# Sau khi đã dùng Genres làm fallback cho Tags thì có thể bỏ Genres.
df = df.drop(columns=["Genres"], errors="ignore")



In [ ]:
df.isnull().sum()

## 11. Check dòng thiếu nhiều thông tin

In [ ]:
# Kiểm tra các game vừa thiếu tag vừa không có mô tả.
# Sửa lỗi precedence: phải đặt (df["Description"] == "No description") trong ngoặc.
display(df[df["Tags"].isnull() & (df["Description"] == "No description")])


## 12. Chốt `Tags` không NULL

In [ ]:
# Chốt Tags không được NaN hoặc chuỗi rỗng trước khi đi vào bước tag mapping.
df["Tags"] = df["Tags"].replace(r"^\s*$", np.nan, regex=True)
df["Tags"] = df["Tags"].fillna("no tags")

log_cleaning_step(
    "05_finalize_tags_before_mapping",
    df,
    "Tags filled from Genres when possible; remaining missing/blank tags set to 'no tags'."
)

display(df["Tags"].isna().sum())
print("Rows with literal empty Tags:", int(df["Tags"].astype(str).str.strip().eq("").sum()))


## 13. Clean text và xóa emoji

In [ ]:
EMOJI_PATTERN = re.compile(
    "["
    "🌀-🗿"  # symbols & pictographs
    "😀-🙏"  # emoticons
    "🚀-🛿"  # transport & map symbols
    "🜀-🝿"
    "🞀-🟿"
    "🠀-🣿"
    "🤀-🧿"
    "🨀-🩯"
    "🩰-🫿"
    "✂-➰"
    "Ⓜ-🉑"
    "]+",
    flags=re.UNICODE,
)


def clean_text(value):
    if pd.isna(value):
        return value

    text = str(value)
    text = EMOJI_PATTERN.sub("", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].apply(clean_text)

# Sau clean text, các field bắt buộc không được rỗng.
for col in ["Name", "Developers", "Publishers", "Tags"]:
    if col in df.columns:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

# Name rỗng thì drop.
df.dropna(subset=["Name"], inplace=True)
df["Name"] = df["Name"].astype(str).str.strip()
df = df[df["Name"].ne("")]

# Dev/Pub rỗng sau clean thì fill qua lại lần cuối, vẫn thiếu thì drop.
df["Publishers"] = df["Publishers"].fillna(df["Developers"])
df["Developers"] = df["Developers"].fillna(df["Publishers"])
df.dropna(subset=["Developers", "Publishers"], inplace=True)

# Tags rỗng thì dùng no tags, không để CSV tạo ô trống.
df["Tags"] = df["Tags"].fillna("no tags")

# Các text optional có fallback rõ ràng.
text_fallbacks = {
    "Description": "No description",
    "Thumbnail": "No thumbnail",
    "Type": "unknown",
    "OsRequirement": "No requirement",
    "MemoryRequirement": "No requirement",
}

for col, fallback in text_fallbacks.items():
    if col in df.columns:
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)
        df[col] = df[col].fillna(fallback)

# Cập nhật lại text_columns sau các bước drop/fill.
text_columns = df.select_dtypes(include="object").columns

df[text_columns].head()



log_cleaning_step(
    "06_clean_text_and_required_fields",
    df,
    "Removed HTML/noisy characters in object columns and revalidated required text fields."
)


In [ ]:
df.isnull().sum()

## 14. Xem cấu trúc DataFrame

In [ ]:
df.info()

## 15. Kiểm tra null bằng `isnull()`

In [ ]:
df.isnull().sum() 

## 16. Normalize price

In [ ]:
df.isna().sum()

## 17. Làm sạch `price`

In [ ]:
df['price'] = (
    df['price']
    .astype(str)
    .str.strip()
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('Free To Play', '0', regex=False)
    .str.replace('Free', '0', regex=False)
    .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
)

df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Không rõ price thì đưa về 0 để tránh NULL ở output cuối.
df['price'] = df['price'].fillna(0)

display(df['price'].describe())
display(df['price'].isna().sum())

log_cleaning_step(
    "07_normalize_price",
    df,
    "Mapped Free/Free To Play to 0 and coerced price to numeric."
)


## 18. Kiểm tra phân phối OSRequirement gốc

In [ ]:
display(df['OsRequirement'].value_counts())

## 19. Normalize OS

In [ ]:
# Chuẩn hóa OSRequirement về các nhóm chính: no requirement, windows, mac, linux, unknown.
def extract_os(x):
    text = str(x).lower().strip()
    text_norm = re.sub(r"[^a-z0-9+.-]+", " ", text)

    no_req_patterns = [
        "no requirement",
        "not specified",
        "n/a",
        "none",
        "any",
        "requires a 64 bit processor",
        "64 bit processor and operating system",
    ]

    if text_norm in {"", "nan", "null"} or any(p in text_norm for p in no_req_patterns):
        return ["no requirement"]

    os_list = []

    windows_patterns = [
        r"\bwindows\b", r"\bwin\s*7\b", r"\bwin\s*8\b", r"\bwin\s*10\b",
        r"\bwin\s*11\b", r"\bwin\s*xp\b", r"\bvista\b", r"\bdirectx\b"
    ]
    mac_patterns = [r"\bmac\b", r"\bmacos\b", r"\bos\s*x\b"]
    linux_patterns = [r"\blinux\b", r"\bubuntu\b", r"\bsteamos\b"]

    if any(re.search(p, text_norm) for p in windows_patterns):
        os_list.append("windows")
    if any(re.search(p, text_norm) for p in mac_patterns):
        os_list.append("mac")
    if any(re.search(p, text_norm) for p in linux_patterns):
        os_list.append("linux")

    # Nếu hỗ trợ đủ 3 hệ điều hành chính thì coi như ít ràng buộc OS.
    if set(os_list) == {"windows", "mac", "linux"}:
        return ["no requirement"]

    if not os_list:
        return ["unknown"]

    # Giữ thứ tự nhưng loại trùng.
    unique_os = []
    for os_name in os_list:
        if os_name not in unique_os:
            unique_os.append(os_name)
    return unique_os

df["OsNew"] = df["OsRequirement"].apply(extract_os)

display(df["OsNew"].value_counts())


## 20. Tách OS

In [ ]:
display(df['OsNew'].value_counts())

## 21. Kiểm tra các OS còn unknown

In [ ]:
df[df['OsNew'].apply(lambda x: 'unknown' in x)]['OsRequirement'].value_counts().head(20)

## 22. Sửa các trường hợp OS unknown còn lại

In [ ]:
# Không ép toàn bộ unknown thành windows.
# Unknown là một nhãn thông tin hợp lệ: nó thể hiện requirement text không đủ rõ để suy luận OS.
# Điều này trung thực hơn so với gán cứng tất cả unknown = windows.
unknown_os_count = int(df["OsNew"].apply(lambda x: "unknown" in x).sum())

os_unknown_audit = (
    df[df["OsNew"].apply(lambda x: "unknown" in x)]
    [["Appid", "Name", "OsRequirement"]]
    .head(100)
)

os_unknown_audit_path = QUALITY_REPORT_DIR / "os_unknown_audit_sample.csv"
os_unknown_audit.to_csv(os_unknown_audit_path, index=False, encoding="utf-8-sig")

print("Unknown OS rows:", unknown_os_count)
print("Saved sample:", os_unknown_audit_path)
display(df["OsNew"].value_counts())
display(os_unknown_audit.head(20))


In [ ]:
display(df['OsNew'].value_counts())

## 23. Thay cột OS gốc bằng cột đã chuẩn hóa

In [ ]:
# Bây giờ ta đã tạo ra nhóm dễ xử lí hơn với 5 nhóm chính, giờ ta sẽ thay vào cột os requirement để tiện cho việc mã hóa sau này
df['OsRequirement'] = df['OsNew']
df.drop(columns=['OsNew'], inplace=True)

In [ ]:
display(df['OsRequirement'].value_counts())

## 24. Check RAM text

In [ ]:
# Tiếp tục là xử lí về ram .
display(df['MemoryRequirement'].value_counts())

## 25. RAM về MB, lỗi thì 0

In [ ]:
def normalize_memory(mem):
    # MemoryRequirement là yêu cầu tối thiểu. Nếu thiếu/không parse được thì coi như không có yêu cầu rõ ràng => 0 MB.
    if pd.isna(mem):
        return 0

    mem = str(mem).lower().strip()
    mem = mem.replace(',', '.')

    if (
        mem == ''
        or 'no requirement' in mem
        or mem in {'none', 'nan', 'null', 'n/a'}
        or 'not specified' in mem
    ):
        return 0

    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(gb|gigs|gigabyte|gigabytes|mb|m\b)', mem)
    values = []

    for num, unit in matches:
        num = float(num)
        if unit in ['gb', 'gigs', 'gigabyte', 'gigabytes']:
            values.append(num * 1024)
        else:
            values.append(num)

    return max(values) if values else 0


df['NewMemoryRequirement'] = df['MemoryRequirement'].apply(normalize_memory)

display(df['NewMemoryRequirement'].value_counts(dropna=False).head(20))
display(df['NewMemoryRequirement'].isna().sum())

## 26. Kiểm tra kết quả RAM sau normalize

In [ ]:
display(df['NewMemoryRequirement'].value_counts())

## 27. Ghi đè RAM đã normalize

In [ ]:
#Vậy ta đã biến feature này về dạng có quy tắc hơn rồi, sau quá trình kiểm tra cũng mang tính hợp lệ nên ta sẽ thay vào cột chính luôn 
df['MemoryRequirement'] = df['NewMemoryRequirement']
df.drop(columns=['NewMemoryRequirement'], inplace=True)

## 28. Kiểm tra CpuRequirement gốc

In [ ]:
display(df['CpuRequirement'].value_counts())

## 29. Normalize CPU

In [ ]:
# Có nhiều cách viết yêu cầu CPU khác nhau. Ta tách thành 3 biến: CPU_req, CPU_GHz, CPU_tier.
def extract_cpu_info(x):
    x = str(x).lower().strip()
    x = x.replace(',', '.')

    cpu_tier = 0
    cpu_ghz = np.nan
    cpu_requirement = 1

    # Không có yêu cầu CPU rõ ràng.
    if (
        x == ''
        or 'no requirement' in x
        or x in {'none', 'nan', 'null', 'n/a'}
        or 'not specified' in x
    ):
        return pd.Series([0, 0, 0])

    # Tách GHz nếu có.
    ghz_match = re.search(r'(\d+(?:\.\d+)?)\s*ghz', x)
    if ghz_match:
        cpu_ghz = float(ghz_match.group(1))

    # Chia cấp CPU theo keyword.
    tiers = []
    if any(k in x for k in ['core2', 'core 2', 'duo', 'pentium', 'dual core', 'dual-core', 'celeron', 'athlon']):
        tiers.append(1)
    if any(k in x for k in ['i3', 'ryzen 3']):
        tiers.append(1)
    if any(k in x for k in ['i5', 'ryzen 5']):
        tiers.append(2)
    if any(k in x for k in ['i7', 'ryzen 7']):
        tiers.append(3)
    if any(k in x for k in ['i9', 'ryzen 9']):
        tiers.append(4)
    if any(k in x for k in ['any', 'almost any', 'multi-core']):
        tiers.append(1)
    if 'last' in x and 'year' in x:
        tiers.append(2)

    cpu_tier = max(tiers) if tiers else 0

    return pd.Series([cpu_requirement, cpu_ghz, cpu_tier])


## 30. Tách CPU requirement

In [ ]:
df[['CPU_req', 'CPU_GHz', 'CPU_tier']] = df['CpuRequirement'].apply(extract_cpu_info)


## 31. Kiểm tra phân phối các cột CPU mới

In [ ]:
display(df['CPU_req'].value_counts())
display(df['CPU_GHz'].value_counts())
display(df['CPU_tier'].value_counts())

In [ ]:
display(df['CPU_GHz'].value_counts())

In [ ]:
df.isnull().sum()

## 32. Fill `CPU_GHz` thiếu

In [ ]:
# Với CPU_GHz bị thiếu, fill bằng median theo CPU_tier.
# Nếu tier đó không có median thì fill 0.
med_ghz = df.groupby('CPU_tier')['CPU_GHz'].median()


def fill_ghz(row):
    if pd.isna(row['CPU_GHz']):
        value = med_ghz.get(row['CPU_tier'], np.nan)
        return 0 if pd.isna(value) else value
    return row['CPU_GHz']


df['CPU_GHz'] = df.apply(fill_ghz, axis=1)
df['CPU_GHz'] = df['CPU_GHz'].fillna(0)

display(df[['CPU_req', 'CPU_GHz', 'CPU_tier']].isna().sum())

In [ ]:
df.isnull().sum()

## 33. Xóa cột CpuRequirement gốc

In [ ]:
#Sau khi tách xong ta chỉ cần bỏ cột CpuRequirement
df.drop(columns=['CpuRequirement'], inplace=True)

## 34. Tags

In [ ]:
def split_items(x):
    if pd.isna(x):
        return []

    items = []
    for item in str(x).split(','):
        item = item.strip().lower()
        item = re.sub(r'\s+', ' ', item)
        if item and item not in items:
            items.append(item)
    return items


def normalize_tag_text(tag):
    tag = normalize_text(tag) if 'normalize_text' in globals() else str(tag).strip().lower()
    tag = re.sub(r'\s+', ' ', tag)
    return tag


# Dùng Series tạm, không tạo cột phụ trong df.
tag_list_raw = df['Tags'].apply(split_items)

display(tag_list_raw.apply(len).describe())
display(tag_list_raw.explode().dropna().value_counts().head(30))


## 35. Rule-based mapping

In [ ]:
tag_rule_mapping = {
    'role-playing': 'rpg',
    'role playing': 'rpg',
    'role-playing game': 'rpg',
    'jrpg': 'rpg',
    'crpg': 'rpg',
    'action rpg': 'rpg',

    'rogue-lite': 'roguelike',
    'rogue like': 'roguelike',
    'rogue-like': 'roguelike',
    'roguelite': 'roguelike',

    'action-adventure': 'action adventure',
    'action adventure game': 'action adventure',

    'hack & slash': 'hack and slash',
    'hack n slash': 'hack and slash',
    'hack-and-slash': 'hack and slash',

    'point & click': 'point and click',
    'point-and-click': 'point and click',

    'free to play': 'free-to-play',
    'free-to-play': 'free-to-play',
    'f2p': 'free-to-play',

    'coop': 'co-op',
    'co op': 'co-op',
    'cooperative': 'co-op',
    'online co-op': 'co-op',
    'local co-op': 'co-op',

    'multi-player': 'multiplayer',
    'multi player': 'multiplayer',
    'online multiplayer': 'multiplayer',

    'single-player': 'singleplayer',
    'single player': 'singleplayer',

    'first-person': 'first person',
    'first person shooter': 'fps',
    'third-person': 'third person',
    'third person shooter': 'third-person shooter',

    'sci fi': 'sci-fi',
    'science fiction': 'sci-fi',

    'turn based': 'turn-based',
    'turn-based strategy': 'strategy',
    'real time strategy': 'rts',
    'real-time strategy': 'rts',

    'massively multiplayer': 'mmo',
    'massively multiplayer online': 'mmo',

    '2 d': '2d',
    '3 d': '3d',
}


def apply_rule_mapping(tags):
    mapped = []
    for tag in tags:
        tag = normalize_tag_text(tag)
        tag = tag_rule_mapping.get(tag, tag).strip()
        if tag and tag not in mapped:
            mapped.append(tag)
    return mapped


tag_list_rule_mapped = tag_list_raw.apply(apply_rule_mapping)
tag_counts_rule = tag_list_rule_mapped.explode().dropna().value_counts()

display(tag_counts_rule.head(50))


## 36. Cài thư viện cho Qwen
Chạy cell này nếu môi trường chưa có các thư viện dưới đây.


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece


## 37. Cấu hình LLM tag mapping

LLM quét toàn bộ tag unique sau clean/rule-based mapping. Không lấy top theo số lượng để làm chuẩn. Nếu tag là đồng nghĩa hoặc là tag con nằm trong một tag mẹ rõ ràng, map về tag mẹ.


In [ ]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# LLM xử lý toàn bộ unique tag sau rule-based mapping.
# Không tạo file mapping trung gian, không lưu tag list trung gian.
LLM_BATCH_SIZE = 6
LLM_MAX_RETRIES = 2
LLM_SLEEP_TIME = 0.2

print("Model:", LLM_MODEL_NAME)
print("LLM batch size:", LLM_BATCH_SIZE)


## 38. Chuẩn bị toàn bộ unique tags cho LLM

In [ ]:
all_unique_tags = sorted(
    str(tag).strip().lower()
    for tag in tag_list_rule_mapped.explode().dropna().unique()
    if str(tag).strip()
)

print("Unique tags for LLM:", len(all_unique_tags))
display(pd.DataFrame({"tag": all_unique_tags}).head(50))


## 39. Load Qwen 7B 4-bit


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("Loaded:", LLM_MODEL_NAME)


## 40. Helper functions cho LLM


In [ ]:
def normalize_text(value):
    if value is None:
        return ""
    return str(value).strip().lower()


def extract_json(text):
    if text is None:
        return {}

    text = str(text).strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {}

    try:
        parsed = json.loads(match.group(0))
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def build_qwen_prompt(batch_tags, all_allowed_tags):
    return f"""
You are normalizing Steam game tags.

Task:
For each input tag, choose exactly one representative tag from the allowed tag list.
Use the parent/broader tag when the input tag is clearly a child/specific form of another tag.
Use the same tag when there is no clear synonym or clear parent tag.

Examples:
- "single-player" -> "singleplayer"
- "role-playing" -> "rpg"
- "rogue-lite" -> "roguelike"
- "first person shooter" -> "fps"
- "turn-based strategy" -> "strategy"
- "online co-op" -> "co-op"

Allowed tags:
{json.dumps(all_allowed_tags, ensure_ascii=False)}

Input tags:
{json.dumps(batch_tags, ensure_ascii=False)}

Rules:
- Return valid JSON only.
- Keys must be the original input tags.
- Values must be one tag from the allowed tags.
- Do not invent new tags.
- Prefer the parent/broader tag if the input tag is clearly contained by that parent tag.
- Prefer the shorter/common representative for synonyms.
- Do not merge unrelated tags.
- Do not over-merge vague themes when the parent relation is unclear.

Output JSON format:
{{
  "input tag": "representative or parent tag"
}}
""".strip()


## 41. Chạy LLM merge candidate tags


In [ ]:
def call_qwen_tag_mapping(batch_tags, all_allowed_tags, max_new_tokens=768, debug=False):
    prompt = build_qwen_prompt(batch_tags, all_allowed_tags)

    messages = [
        {
            "role": "system",
            "content": "You are a strict JSON generator. Return only valid JSON and nothing else.",
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output[0][inputs.input_ids.shape[-1]:]
    generated = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    if debug:
        print("=" * 80)
        print("BATCH:", batch_tags)
        print("RAW GENERATED:")
        print(repr(generated[:2000]))

    parsed = extract_json(generated)
    allowed_set = set(all_allowed_tags)
    cleaned_mapping = {}

    for tag in batch_tags:
        tag_norm = normalize_text(tag)
        value = parsed.get(tag, parsed.get(tag_norm, tag_norm))
        value_norm = normalize_text(value)

        # Nếu model trả tag ngoài allowed list thì giữ chính nó.
        cleaned_mapping[tag_norm] = value_norm if value_norm in allowed_set else tag_norm

    return cleaned_mapping


def run_qwen_mapping(
    tags_for_llm,
    all_allowed_tags,
    batch_size=6,
    sleep_time=0.2,
    max_retries=2,
    debug=False,
):
    tags_for_llm = [normalize_text(tag) for tag in tags_for_llm if normalize_text(tag)]
    all_allowed_tags = [normalize_text(tag) for tag in all_allowed_tags if normalize_text(tag)]

    mapping = {}
    total = len(tags_for_llm)

    for start in range(0, total, batch_size):
        batch = tags_for_llm[start:start + batch_size]
        batch_mapping = {}

        for attempt in range(max_retries + 1):
            try:
                batch_mapping = call_qwen_tag_mapping(
                    batch_tags=batch,
                    all_allowed_tags=all_allowed_tags,
                    debug=debug and attempt == 0,
                )
                if batch_mapping:
                    break

                print(f"Empty JSON batch {start // batch_size + 1}, retry {attempt + 1}/{max_retries}")

            except Exception as exc:
                print(f"Failed batch {start // batch_size + 1}, retry {attempt + 1}/{max_retries}: {exc}")

            time.sleep(sleep_time)

        for tag in batch:
            tag_norm = normalize_text(tag)
            mapping[tag_norm] = batch_mapping.get(tag_norm, tag_norm)

        print(f"Done {min(start + batch_size, total)}/{total}")
        time.sleep(sleep_time)

    return mapping


def resolve_mapping(mapping, max_depth=8):
    resolved = {}

    for raw_tag in mapping.keys():
        current = normalize_text(raw_tag)
        seen = set()

        for _ in range(max_depth):
            next_tag = normalize_text(mapping.get(current, current))

            if not next_tag or next_tag in seen or next_tag == current:
                break

            seen.add(current)
            current = next_tag

        resolved[normalize_text(raw_tag)] = current

    return resolved


## 42. Load và tạo mapping


In [ ]:
tag_mapping_llm = run_qwen_mapping(
    tags_for_llm=all_unique_tags,
    all_allowed_tags=all_unique_tags,
    batch_size=LLM_BATCH_SIZE,
    sleep_time=LLM_SLEEP_TIME,
    max_retries=LLM_MAX_RETRIES,
    debug=False,
)

# Resolve chuỗi mapping kiểu A -> B -> C thành A -> C.
tag_mapping_llm = resolve_mapping(tag_mapping_llm)

print("LLM mapping size:", len(tag_mapping_llm))
print("Changed mappings:", sum(raw != mapped for raw, mapped in tag_mapping_llm.items()))

display(
    pd.DataFrame(list(tag_mapping_llm.items()), columns=["raw_tag", "mapped_tag"])
    .query("raw_tag != mapped_tag")
    .head(100)
)


## 43. Tạo cột tags


In [ ]:
def apply_llm_mapping(tags):
    mapped_tags = []

    for tag in tags:
        tag_norm = normalize_text(tag)
        mapped = tag_mapping_llm.get(tag_norm, tag_norm)
        mapped_norm = normalize_text(mapped)

        if mapped_norm and mapped_norm not in mapped_tags:
            mapped_tags.append(mapped_norm)

    return mapped_tags


mapped_tags_series = tag_list_rule_mapped.apply(apply_llm_mapping)

# Chỉ tạo đúng một cột mới là tags.
# Không lọc top tag theo frequency, không tạo feature phụ trong df.
df['tags'] = mapped_tags_series.apply(
    lambda tags: ', '.join(tags) if len(tags) > 0 else 'no tags'
)

df['tags'] = df['tags'].fillna('no tags')
df.loc[df['tags'].astype(str).str.strip().eq(''), 'tags'] = 'no tags'

print('Unique raw/rule tags:', tag_list_rule_mapped.explode().dropna().nunique())
print('Unique tags after LLM parent/synonym merge:', mapped_tags_series.explode().dropna().nunique())
print('Rows with empty tags:', int(df['tags'].astype(str).str.strip().eq('').sum()))

display(
    pd.DataFrame(list(tag_mapping_llm.items()), columns=['raw_tag', 'mapped_tag'])
    .query('raw_tag != mapped_tag')
    .head(100)
)
display(df[['Tags', 'tags']].head(20))


In [ ]:
# Sau khi map thì drop cột Tags ban đầu 
df = df.drop('Tags', axis=1)


## 44. Save


In [ ]:
# ============================================================
# FINAL DATASET VALIDATION + SAVE + RELOAD TEST
# ============================================================
# Check cả NaN lẫn chuỗi rỗng, vì chuỗi rỗng khi save CSV rồi đọc lại bằng
# pd.read_csv() có thể thành NaN trên Kaggle/notebook khác.
# ============================================================

# Đánh lại Rank lần cuối sau tất cả thao tác drop/filter.
df = df.reset_index(drop=True)
df["Rank"] = np.arange(1, len(df) + 1)

# Appid phải unique ở dataset cuối.
if not df["Appid"].is_unique:
    duplicate_final = (
        df[df["Appid"].duplicated(keep=False)]
        .sort_values(["Appid", "Rank"])
        .copy()
    )
    duplicate_final_path = QUALITY_REPORT_DIR / "duplicate_appid_final_error.csv"
    duplicate_final.to_csv(duplicate_final_path, index=False, encoding="utf-8-sig")
    display(duplicate_final[["Appid", "Name", "Rank"]].head(50))
    raise ValueError(f"Dataset cuối vẫn còn duplicate Appid. Audit saved: {duplicate_final_path}")

# Rank phải liên tục.
if not (df["Rank"].is_unique and df["Rank"].min() == 1 and df["Rank"].max() == len(df)):
    raise ValueError("Rank cuối không unique hoặc không liên tục từ 1 đến N.")

null_counts = df.isna().sum()
null_counts = null_counts[null_counts > 0]

empty_counts = df.select_dtypes(include="object").apply(
    lambda s: s.astype(str).str.strip().eq("").sum()
)
empty_counts = empty_counts[empty_counts > 0]

if len(null_counts) > 0:
    display(null_counts)
    raise ValueError("Dataset vẫn còn NaN trước khi save.")

if len(empty_counts) > 0:
    display(empty_counts)
    raise ValueError("Dataset vẫn còn chuỗi rỗng trước khi save.")

log_cleaning_step(
    "08_final_before_save",
    df,
    "Final dataset passed NaN, empty-string, duplicate Appid and continuous Rank checks."
)

print("OK: Không còn NaN hoặc chuỗi rỗng trước khi save.")
print("OK: Appid unique =", df["Appid"].is_unique)
print("OK: Rank continuous = 1..N")

# Save final processed dataset.
df.to_csv(PROCESSED_PATH, index=False, encoding="utf-8-sig")
print("Saved final processed dataset:", PROCESSED_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Test lại file CSV sau khi save và đọc lại bằng mặc định của pandas/Kaggle.
test_df = pd.read_csv(PROCESSED_PATH)

reload_null_counts = test_df.isna().sum()
reload_null_counts = reload_null_counts[reload_null_counts > 0]

reload_empty_counts = test_df.select_dtypes(include="object").apply(
    lambda s: s.astype(str).str.strip().eq("").sum()
)
reload_empty_counts = reload_empty_counts[reload_empty_counts > 0]

if len(reload_null_counts) > 0:
    display(reload_null_counts)
    raise ValueError("File CSV sau khi đọc lại vẫn sinh NaN.")

if len(reload_empty_counts) > 0:
    display(reload_empty_counts)
    raise ValueError("File CSV sau khi đọc lại vẫn còn chuỗi rỗng.")

if "Appid" in test_df.columns and test_df["Appid"].duplicated().sum() > 0:
    display(test_df[test_df["Appid"].duplicated(keep=False)].sort_values("Appid").head(50))
    raise ValueError("File CSV sau khi đọc lại vẫn còn duplicate Appid.")

print("OK: CSV reload không còn NaN.")
print("OK: CSV reload không còn chuỗi rỗng.")
print("OK: CSV reload Appid unique.")

# ------------------------------------------------------------
# Export cleaning audit files for report.
# ------------------------------------------------------------

cleaning_audit_df = pd.DataFrame(cleaning_step_stats).drop_duplicates("step", keep="last")
cleaning_audit_path = QUALITY_REPORT_DIR / "cleaning_step_audit.csv"
cleaning_audit_df.to_csv(cleaning_audit_path, index=False, encoding="utf-8-sig")

final_quality_summary = pd.DataFrame([
    {"metric": "raw_rows", "value": len(df_raw_snapshot)},
    {"metric": "final_rows", "value": len(df)},
    {"metric": "rows_removed_total", "value": len(df_raw_snapshot) - len(df)},
    {"metric": "raw_duplicate_appid", "value": int(df_raw_snapshot["Appid"].duplicated().sum()) if "Appid" in df_raw_snapshot.columns else "N/A"},
    {"metric": "final_duplicate_appid", "value": int(df["Appid"].duplicated().sum())},
    {"metric": "final_missing_cells_before_save", "value": int(df.isna().sum().sum())},
    {"metric": "final_empty_string_cells_before_save", "value": count_empty_string_cells(df)},
    {"metric": "reload_missing_cells", "value": int(test_df.isna().sum().sum())},
    {"metric": "reload_empty_string_cells", "value": count_empty_string_cells(test_df)},
    {"metric": "reload_duplicate_appid", "value": int(test_df["Appid"].duplicated().sum()) if "Appid" in test_df.columns else "N/A"},
])
final_quality_summary_path = QUALITY_REPORT_DIR / "final_cleaning_quality_summary.csv"
final_quality_summary.to_csv(final_quality_summary_path, index=False, encoding="utf-8-sig")

# Bảng claim -> evidence để copy vào báo cáo.
claim_evidence_cleaning = pd.DataFrame([
    {
        "claim": "Dataset cuối unique theo Appid.",
        "evidence": f"final_duplicate_appid = {int(df['Appid'].duplicated().sum())}",
        "report_usage": "Dùng để chứng minh Appid là khóa định danh đáng tin cậy."
    },
    {
        "claim": "Rank đã được realign sau khi xử lý trùng crawl.",
        "evidence": f"Rank min={df['Rank'].min()}, max={df['Rank'].max()}, unique={df['Rank'].is_unique}",
        "report_usage": "Dùng để giải thích lỗi rank gối đầu do crawl ngắt quãng."
    },
    {
        "claim": "Dataset cuối không còn NaN hoặc chuỗi rỗng trước khi save.",
        "evidence": f"missing={int(df.isna().sum().sum())}, empty_string={count_empty_string_cells(df)}",
        "report_usage": "Dùng cho phần Data Quality Validation."
    },
    {
        "claim": "File CSV sau khi đọc lại vẫn ổn định.",
        "evidence": f"reload_missing={int(test_df.isna().sum().sum())}, reload_empty_string={count_empty_string_cells(test_df)}",
        "report_usage": "Dùng để chứng minh dataset public không phát sinh NULL do empty string."
    },
    {
        "claim": "Pipeline có audit cho duplicate Appid.",
        "evidence": str(QUALITY_REPORT_DIR / "duplicate_appid_audit_before_dedup.csv"),
        "report_usage": "Dùng khi thầy hỏi vì sao raw có trùng và nhóm xử lý thế nào."
    },
])
claim_evidence_cleaning_path = QUALITY_REPORT_DIR / "cleaning_claim_evidence.csv"
claim_evidence_cleaning.to_csv(claim_evidence_cleaning_path, index=False, encoding="utf-8-sig")


In [ ]:
df.isna().sum()
